# STT API — Colab 실행 노트북

자체 호스팅과 **동일한 코드**를 Google Colab(Pro)에서 실행한다.
저장소를 클론해 의존성을 설치하고, GPU에서 모델을 프리로드한 뒤 API 서버를 띄운다.

## 준비
1. 런타임 → 런타임 유형 변경 → **GPU** 선택 (T4 이상)
2. 아래 셀을 위에서부터 실행
3. 외부에서 접속하려면 cloudflared 임시 터널의 공개 URL을 사용 (아래 출력)

> Colab은 데모/개발용이다. 세션 종료 시 서버가 멈추고, 터널 URL은 공개된다.
> 운영은 `run.sh`로 자체 호스팅하고, 공개 URL에는 반드시 `STT_API_KEY`를 설정한다.

In [8]:
!nvidia-smi || echo 'GPU 없음 - CPU로 실행됩니다(느림)'
import platform, sys
print(platform.platform(), sys.version.split()[0])

MODEL = 'small'   # T4: small / L4·A100: 'large-v3-turbo'
PORT = 8000
print('model:', MODEL)

Sun Nov 22 06:16:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os, shutil, subprocess

# 셀을 여러 번 실행해도 같은 위치에서 시작한다 (중첩 chdir 방지)
WORK_ROOT = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(WORK_ROOT)

REPO_URL = 'https://github.com/lukehemmin/STT_API.git'
REPO_DIR = 'STT_API'
REQUIRED = ['requirements.txt', 'app/main.py', 'app/engine.py']
REPO_PATH = os.path.join(WORK_ROOT, REPO_DIR)

def clone():
    shutil.rmtree(REPO_PATH, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_PATH], check=True)

if not os.path.isdir(os.path.join(REPO_PATH, '.git')):
    clone()
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '--ff-only'], check=False)
    if not os.path.exists(os.path.join(REPO_PATH, 'requirements.txt')):
        print('기존 클론이 오래되어 새로 클론합니다.')
        clone()

os.chdir(REPO_PATH)
missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    raise RuntimeError('클론한 저장소에 코드가 없습니다: ' + ', '.join(missing))
print('cwd:', os.getcwd())
print('HEAD:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                               capture_output=True, text=True).stdout.strip())


In [ ]:
import shutil, subprocess, sys

def run(cmd):
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print('실패:', ' '.join(cmd))
        print((res.stderr or res.stdout)[-4000:])
        raise RuntimeError('명령 실패: ' + ' '.join(cmd))
    return res

subprocess.run(['apt-get', '-qq', 'update'], capture_output=True)  # 경고만 나오는 경우가 있어 무시
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])

try:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
except RuntimeError:
    print('\n[힌트] Python ' + sys.version.split()[0] + ' 에서 의존성 설치가 실패했습니다.')
    print('  - 이 Python 버전용 휠이 없을 수 있습니다. 최신 빌드를 시도해 보세요:')
    print('      pip install -U ctranslate2 faster-whisper')
    print('  - 그래도 실패하면 런타임 → 런타임 유형 변경에서 Python 3.11 로 바꿔 다시 실행하세요.')
    raise

import importlib.metadata as md
for pkg in ('faster-whisper', 'ctranslate2'):
    try:
        print(pkg, md.version(pkg))
    except Exception:
        print(pkg, '(버전 확인 실패)')

import ctranslate2
print('CUDA device count:', ctranslate2.get_cuda_device_count())
print('ffmpeg:', shutil.which('ffmpeg'))
print('의존성 설치 완료')


In [11]:
import os, subprocess

res = subprocess.run(['python', '-m', 'app.preload'], capture_output=True, text=True,
                     env=dict(os.environ, STT_MODEL=MODEL))
print(res.stdout.strip())

if res.returncode != 0:
    err = res.stderr or ''
    print(err[-4000:])
    if any(k in err.lower() for k in ('cublas', 'cudnn', 'libcu', 'no kernel image')):
        print('\\n[힌트] GPU 라이브러리 문제로 보입니다. 아래 중 하나를 선택하세요.')
        print("  1) CPU로 계속: 셀 5의 env 에 STT_DEVICE='cpu' 를 추가하고 실행 (매우 느림)")
        print('  2) CUDA 라이브러리 설치 후 재시도: !pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12')
        print('  3) 더 작은 모델: MODEL = "base" 로 바꿔 다시 실행')
    raise RuntimeError('모델 프리로드 실패')
print('프리로드 완료:', MODEL)


model: small
device: cuda
compute_type: float16
loaded in 7.3s
프리로드 완료: small


In [12]:
import os, re, subprocess, time, requests

env = dict(os.environ, STT_MODEL=MODEL, STT_DEVICE='auto', STT_PRELOAD='1')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', str(PORT)],
    env=env,
    stdout=open('server.log', 'w'),
    stderr=subprocess.STDOUT,
)

for _ in range(90):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/healthz', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('서버 시작 실패\n' + open('server.log').read()[-3000:])

print('server:', requests.get(f'http://127.0.0.1:{PORT}/v1/info').json())

# 외부 접근용 임시 터널 (공개 URL이므로 민감한 데이터 업로드 금지)
PUBLIC_URL = None
CF = '/usr/local/bin/cloudflared'
if not os.path.exists(CF):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CF,
    ], check=True)
    os.chmod(CF, 0o755)

tunnel = subprocess.Popen(
    [CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=open('cloudflared.log', 'w'),
    stderr=subprocess.STDOUT,
)
for _ in range(45):
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cloudflared.log').read())
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(1)

print('PUBLIC_URL:', PUBLIC_URL or '(터널 생성 실패 - 로컬에서만 사용)')

server: {'engine': {'model': 'small', 'loaded': True, 'device': 'cuda', 'compute_type': 'float16', 'language': None}, 'limits': {'max_upload_mb': 2048, 'max_active_jobs': 4, 'max_active_streams': 4, 'max_stream_ms': 7200000, 'stream_sample_rate': 16000, 'stream_encoding': 'pcm_s16le', 'stream_channels': 1}, 'active_jobs': 0, 'active_streams': 0}
PUBLIC_URL: https://aka-singer-well-briefing.trycloudflare.com


In [ ]:
import os, time, requests

API = f'http://127.0.0.1:{PORT}'
WAV = 'jfk.wav'
if not os.path.exists(WAV) or os.path.getsize(WAV) < 1024:
    !wget -q -O jfk.wav https://github.com/ggerganov/whisper.cpp/raw/master/samples/jfk.wav
print('wav:', os.path.getsize(WAV), 'bytes')

with open(WAV, 'rb') as fh:
    res = requests.post(
        f'{API}/v1/transcriptions',
        files={'file': ('jfk.wav', fh, 'audio/wav')},
        data={'language': 'en'},
    )
print(res.status_code, res.text[:400])
if res.status_code != 202:
    raise RuntimeError('잡 생성 실패')
job_id = res.json()['id']

for _ in range(180):
    st = requests.get(f'{API}/v1/transcriptions/{job_id}').json()
    print('status:', st['status'], '| progress:', st['progress'], '| error:', st.get('error'))
    if st['status'] in {'completed', 'failed', 'canceled'}:
        break
    time.sleep(1)

r = requests.get(f'{API}/v1/transcriptions/{job_id}/result')
if r.status_code != 200:
    print('result 실패:', r.status_code)
    print(r.text[:1000])          # 실패 원인 확인용
    raise RuntimeError('결과 조회 실패: ' + str(r.status_code))

print('TEXT:', r.json()['text'])
print(requests.get(f'{API}/v1/transcriptions/{job_id}/result', params={'format': 'srt'}).text)


In [ ]:
import asyncio, json, subprocess, websockets

def to_pcm(path):
    return subprocess.run(
        ['ffmpeg', '-v', 'error', '-i', path, '-f', 's16le', '-ac', '1', '-ar', '16000', '-'],
        check=True, capture_output=True,
    ).stdout

async def stream_demo():
    pcm = to_pcm('jfk.wav')
    step = 3200  # 100 ms, 16-bit mono
    async with websockets.connect(f'ws://127.0.0.1:{PORT}/v1/stream?language=en') as ws:
        print(json.loads(await ws.recv()))  # ready

        async def sender():
            for i in range(0, len(pcm), step):
                await ws.send(pcm[i:i + step])
                await asyncio.sleep(0.1)  # 실시간 속도
            await ws.send(json.dumps({'type': 'end'}))

        async def receiver():
            async for msg in ws:
                ev = json.loads(msg)
                if ev['type'] in {'partial', 'final'}:
                    print(f"[{ev['type']:7}] {ev['text']}")
                elif ev['type'] == 'done':
                    print('done:', ev)
                    break
                elif ev['type'] == 'error':
                    print('error:', ev)

        await asyncio.gather(sender(), receiver())

await stream_demo()

In [ ]:
print(open('server.log').read()[-2000:])

# 정리
# server.terminate(); tunnel.terminate()

## 참고

- API 문서: 서버 실행 후 `/docs`
- 모델 변경: `MODEL` 값을 바꾸고 프리로드/서버 셀을 다시 실행
- 인증: `STT_API_KEY=...` 환경 변수를 주면 HTTP는 `Authorization: Bearer`, WS는 `?api_key=`가 필요
- 세션 한도: `STT_MAX_ACTIVE_STREAMS`, `STT_MAX_ACTIVE_JOBS`
- Colab은 세션/유휴 시간 제한이 있고 영구 운영에 적합하지 않다. 운영은 `./run.sh` 사용.